## Libraries and Packages

In [1]:
import os
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.1/239.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

In [ ]:
train_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_train_file.xlsx')
eval_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_valid_file.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_test_file.xlsx')

In [ ]:
train_df.head()

,text,author,label
0,However the hardware simulated by your virtual...,1,0
1,Note: Looks like VirtualBox's website is tempo...,2,0
2,Modern CI/CD pipelines leverage infrastructure...,5,1
3,"Generative AI models, like large language mode...",1,0
4,"RAID configurations, such as RAID 10, provide ...",2,0


### Total Samples Distribution

In [ ]:
# Function to count rows per author for a given DataFrame
def count_rows_per_author(df, dataset_name):
    author_counts = df['author'].value_counts().reset_index()
    author_counts.columns = ['author', 'count']
    author_counts['dataset'] = dataset_name  # Add a column to identify the dataset
    return author_counts

# Get row counts for each DataFrame
train_counts = count_rows_per_author(train_df, "Train")
eval_counts = count_rows_per_author(eval_df, "Validation")
test_counts = count_rows_per_author(test_df, "Test")

# Combine all counts into a single DataFrame
combined_counts = pd.concat([train_counts, eval_counts, test_counts])

# Create a grouped bar chart
fig = px.bar(combined_counts, x='author', y='count', color='dataset', barmode='group',
             labels={'author': 'Author', 'count': 'Number of Rows'},
             title="Total Number of Rows per Author Across All DataFrames",
             text='count')

# Format and position the text
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Author',
    yaxis_title='Number of Rows',
    legend_title='Dataset'
)

# Show the plot
fig.show()

### Dublicate Rows, Authors & DF

In [ ]:
# Function to find duplicated text for each author and count them
def find_duplicated_text_counts(df):
    # Group by 'author' and 'text', then count occurrences
    grouped = df.groupby(['author', 'text']).size().reset_index(name='count')

    # Filter rows where count > 1 (duplicated text for the author)
    duplicated_texts = grouped[grouped['count'] > 1]

    # Group by 'author' and count the number of duplicated texts
    duplicated_counts = duplicated_texts.groupby('author').size().reset_index(name='duplicated_count')

    return duplicated_counts

# Find duplicated text counts for each DataFrame
train_duplicates = find_duplicated_text_counts(train_df)
eval_duplicates = find_duplicated_text_counts(eval_df)
test_duplicates = find_duplicated_text_counts(test_df)

# Add a 'dataset' column to identify the source DataFrame
train_duplicates['dataset'] = 'train'
eval_duplicates['dataset'] = 'eval'
test_duplicates['dataset'] = 'test'

# Combine the results into a single DataFrame
combined_duplicates = pd.concat([train_duplicates, eval_duplicates, test_duplicates])

# Get the list of all unique authors across all datasets
all_authors = pd.concat([train_df, eval_df, test_df])['author'].unique()

# Ensure all authors are included in the combined_duplicates DataFrame
# Create a DataFrame with all authors and datasets, then merge with combined_duplicates
all_authors_df = pd.DataFrame({
    'author': all_authors
})

# Create a cartesian product of all authors and datasets
all_combinations = pd.MultiIndex.from_product(
    [all_authors, ['train', 'eval', 'test']],
    names=['author', 'dataset']
).to_frame(index=False)

# Merge with combined_duplicates to fill missing values with 0
combined_duplicates = all_combinations.merge(
    combined_duplicates,
    on=['author', 'dataset'],
    how='left'
).fillna({'duplicated_count': 0})

# Plot the count of duplicated texts using Plotly
fig = px.bar(combined_duplicates, x='author', y='duplicated_count', color='dataset', barmode='group',
             text='duplicated_count', title='Count of Duplicated Texts by Author and Dataset')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Author',
    yaxis_title='Count of Duplicated Texts',
    legend_title='Dataset'
)

# Show the plot
fig.show()

In [ ]:
# Function to find and display two duplicate samples for each author
def find_and_display_duplicates(df, dataset_name):
    print(f"\nDuplicate samples for {dataset_name} dataset:")

    # Group by 'author' and 'text' to find duplicates
    grouped = df.groupby(['author', 'text']).size().reset_index(name='count')

    # Filter rows where count > 1 (duplicates)
    duplicates = grouped[grouped['count'] > 1]

    # Iterate over each author and display two duplicate samples
    for author in df['author'].unique():
        author_duplicates = duplicates[duplicates['author'] == author]

        if not author_duplicates.empty:
            print(f"\nAuthor: {author}")
            for _, row in author_duplicates.head(2).iterrows():  # Limit to 2 duplicates
                duplicate_text = row['text']
                duplicate_indices = df[(df['author'] == author) & (df['text'] == duplicate_text)].index.tolist()
                print(f"Duplicate text: '{duplicate_text}'")
                print(f"Indices: {duplicate_indices}")
        else:
            print(f"\nAuthor: {author} has no duplicates.")

# Find and display duplicates for each DataFrame
find_and_display_duplicates(train_df, "train")
find_and_display_duplicates(eval_df, "eval")
find_and_display_duplicates(test_df, "test")


Duplicate samples for train dataset:

Author: 1
Duplicate text: '0'
Indices: [38495, 45969]
Duplicate text: 'Containerization technologies like Docker and Kubernetes simplify application deployment and management by packaging applications and their dependencies into isolated containers.  This improves portability and scalability across various environments.'
Indices: [30067, 52320]

Author: 2
Duplicate text: 'Containerization technologies like Docker and Kubernetes simplify application deployment and management across different environments.  Their lightweight nature and efficient resource utilization make them ideal for microservices architectures and cloud-native applications.'
Indices: [22849, 25005, 44809]
Duplicate text: 'Containerization technologies like Docker and Kubernetes simplify application deployment and management across various environments.  Their lightweight nature and efficient resource utilization contribute to cost savings and improved scalability in cloud-based d

### Drop Dublicates

In [ ]:
# Function to delete duplicates while keeping the first occurrence
def delete_duplicates(df):
    # Drop duplicates based on 'author' and 'text', keeping the first occurrence
    df = df.drop_duplicates(subset=['author', 'text'], keep='first')
    return df

# Delete duplicates in all DataFrames
train_df = delete_duplicates(train_df)
eval_df = delete_duplicates(eval_df)
test_df = delete_duplicates(test_df)

In [ ]:
# Function to find duplicated text for each author and count them
def find_duplicated_text_counts(df):
    # Group by 'author' and 'text', then count occurrences
    grouped = df.groupby(['author', 'text']).size().reset_index(name='count')

    # Filter rows where count > 1 (duplicated text for the author)
    duplicated_texts = grouped[grouped['count'] > 1]

    # Group by 'author' and count the number of duplicated texts
    duplicated_counts = duplicated_texts.groupby('author').size().reset_index(name='duplicated_count')

    return duplicated_counts

# Find duplicated text counts for each DataFrame
train_duplicates = find_duplicated_text_counts(train_df)
eval_duplicates = find_duplicated_text_counts(eval_df)
test_duplicates = find_duplicated_text_counts(test_df)

# Add a 'dataset' column to identify the source DataFrame
train_duplicates['dataset'] = 'train'
eval_duplicates['dataset'] = 'eval'
test_duplicates['dataset'] = 'test'

# Combine the results into a single DataFrame
combined_duplicates = pd.concat([train_duplicates, eval_duplicates, test_duplicates])

# Get the list of all unique authors across all datasets
all_authors = pd.concat([train_df, eval_df, test_df])['author'].unique()

# Ensure all authors are included in the combined_duplicates DataFrame
# Create a DataFrame with all authors and datasets, then merge with combined_duplicates
all_authors_df = pd.DataFrame({
    'author': all_authors
})

# Create a cartesian product of all authors and datasets
all_combinations = pd.MultiIndex.from_product(
    [all_authors, ['train', 'eval', 'test']],
    names=['author', 'dataset']
).to_frame(index=False)

# Merge with combined_duplicates to fill missing values with 0
combined_duplicates = all_combinations.merge(
    combined_duplicates,
    on=['author', 'dataset'],
    how='left'
).fillna({'duplicated_count': 0})

# Plot the count of duplicated texts using Plotly
fig = px.bar(combined_duplicates, x='author', y='duplicated_count', color='dataset', barmode='group',
             text='duplicated_count', title='Count of Duplicated Texts by Author and Dataset')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Author',
    yaxis_title='Count of Duplicated Texts',
    legend_title='Dataset'
)

# Show the plot
fig.show()

### Drop Author

In [ ]:
if 'author' in train_df.columns:
  train_df.drop(columns=['author'], inplace=True)
if 'author' in eval_df.columns:
  eval_df.drop(columns=['author'], inplace=True)
if 'author' in test_df.columns:
  test_df.drop(columns=['author'], inplace=True)

train_df.head()

,text,label
0,However the hardware simulated by your virtual...,0
1,Note: Looks like VirtualBox's website is tempo...,0
2,Modern CI/CD pipelines leverage infrastructure...,1
3,"Generative AI models, like large language mode...",0
4,"RAID configurations, such as RAID 10, provide ...",0


### Nulls or NaN etc

In [ ]:
# Define a list of garbage values (customize as needed)
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

# Function to find nulls, NaNs, and garbage values in the 'text' column
def find_garbage_values(df, dataset_name):
    print(f"\n=======================Checking for garbage values in {dataset_name} dataset ==========================")

    # Check for nulls or NaNs
    null_mask = df['text'].isna()
    null_rows = df[null_mask]
    print(f"Null or NaN values:\n{null_rows}\n")

    # Check for empty strings or whitespace-only strings
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()
    empty_rows = df[empty_mask]
    print(f"Empty or whitespace-only strings:\n{empty_rows}\n")

    # Check for other garbage values
    garbage_mask = df['text'].isin(garbage_values)
    garbage_rows = df[garbage_mask]
    print(f"Other garbage values:\n{garbage_rows}\n")

    # Combine all masks to get rows with any garbage value
    combined_mask = null_mask | empty_mask | garbage_mask
    combined_rows = df[combined_mask]
    print(f"All rows with garbage values:\n{combined_rows}\n")

    return combined_rows

# Find garbage values in each DataFrame
train_garbage = find_garbage_values(train_df, "Train")
eval_garbage = find_garbage_values(eval_df, "Validation")
test_garbage = find_garbage_values(test_df, "Test")


=======================Checking for garbage values in Train dataset ==========================
Null or NaN values:
Empty DataFrame
Columns: [text, label]
Index: []

Empty or whitespace-only strings:
Empty DataFrame
Columns: [text, label]
Index: []

Other garbage values:
Empty DataFrame
Columns: [text, label]
Index: []

All rows with garbage values:
Empty DataFrame
Columns: [text, label]
Index: []


=======================Checking for garbage values in Validation dataset ==========================
Null or NaN values:
Empty DataFrame
Columns: [text, label]
Index: []

Empty or whitespace-only strings:
Empty DataFrame
Columns: [text, label]
Index: []

Other garbage values:
Empty DataFrame
Columns: [text, label]
Index: []

All rows with garbage values:
Empty DataFrame
Columns: [text, label]
Index: []


=======================Checking for garbage values in Test dataset ==========================
Null or NaN values:
Empty DataFrame
Columns: [text, label]
Index: []

Empty or whitespace-only s

In [ ]:
# Define a list of garbage values (customize as needed)
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

# Function to find nulls, NaNs, and garbage values in the 'text' column
def find_garbage_values(df):
    # Check for nulls or NaNs
    null_mask = df['text'].isna()

    # Check for empty strings or whitespace-only strings
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()

    # Check for other garbage values
    garbage_mask = df['text'].isin(garbage_values)

    # Combine all masks to get rows with any garbage value
    combined_mask = null_mask | empty_mask | garbage_mask
    combined_rows = df[combined_mask]

    return combined_rows

# Find garbage rows in each DataFrame
train_garbage = find_garbage_values(train_df)
eval_garbage = find_garbage_values(eval_df)
test_garbage = find_garbage_values(test_df)

# Count the number of garbage rows in each DataFrame
garbage_counts = {
    'train': len(train_garbage),
    'eval': len(eval_garbage),
    'test': len(test_garbage)
}

# Convert the counts to a DataFrame for plotting
garbage_counts_df = pd.DataFrame({
    'dataset': list(garbage_counts.keys()),
    'garbage_count': list(garbage_counts.values())
})

# Plot the count of garbage rows using Plotly
fig = px.bar(garbage_counts_df, x='dataset', y='garbage_count',
             labels={'dataset': 'Dataset', 'garbage_count': 'Count of Garbage Rows'},
             title='Count of Garbage Rows in Each Dataset',
             text='garbage_count')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Dataset',
    yaxis_title='Count of Garbage Rows',
    showlegend=False
)

# Show the plot
fig.show()

In [ ]:
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

def drop_garbage_values(df):
    null_mask = df['text'].isna()
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()
    garbage_mask = df['text'].isin(garbage_values)
    combined_mask = null_mask | empty_mask | garbage_mask
    cleaned_df = df[~combined_mask]

    return cleaned_df

train_df = drop_garbage_values(train_df)
eval_df = drop_garbage_values(eval_df)
test_df = drop_garbage_values(test_df)

In [ ]:
# Define a list of garbage values (customize as needed)
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

# Function to find nulls, NaNs, and garbage values in the 'text' column
def find_garbage_values(df):
    # Check for nulls or NaNs
    null_mask = df['text'].isna()

    # Check for empty strings or whitespace-only strings
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()

    # Check for other garbage values
    garbage_mask = df['text'].isin(garbage_values)

    # Combine all masks to get rows with any garbage value
    combined_mask = null_mask | empty_mask | garbage_mask
    combined_rows = df[combined_mask]

    return combined_rows

# Find garbage rows in each DataFrame
train_garbage = find_garbage_values(train_df)
eval_garbage = find_garbage_values(eval_df)
test_garbage = find_garbage_values(test_df)

# Count the number of garbage rows in each DataFrame
garbage_counts = {
    'train': len(train_garbage),
    'eval': len(eval_garbage),
    'test': len(test_garbage)
}

# Convert the counts to a DataFrame for plotting
garbage_counts_df = pd.DataFrame({
    'dataset': list(garbage_counts.keys()),
    'garbage_count': list(garbage_counts.values())
})

# Plot the count of garbage rows using Plotly
fig = px.bar(garbage_counts_df, x='dataset', y='garbage_count',
             labels={'dataset': 'Dataset', 'garbage_count': 'Count of Garbage Rows'},
             title='Count of Garbage Rows in Each Dataset',
             text='garbage_count')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Dataset',
    yaxis_title='Count of Garbage Rows',
    showlegend=False
)

# Show the plot
fig.show()

### Check Data Leakage

In [ ]:
train_texts = set(train_df['text'])
valid_texts = set(eval_df['text'])
test_texts = set(test_df['text'])

# Check for common samples
common_train_valid = train_texts.intersection(valid_texts)
common_train_test = train_texts.intersection(test_texts)
common_valid_test = valid_texts.intersection(test_texts)

print(f"🔍 Common samples between Train & Validation: {len(common_train_valid)}")
print(f"🔍 Common samples between Train & Test: {len(common_train_test)}")
print(f"🔍 Common samples between Validation & Test: {len(common_valid_test)}")

🔍 Common samples between Train & Validation: 176
🔍 Common samples between Train & Test: 162
🔍 Common samples between Validation & Test: 77


In [ ]:
eval_df = eval_df[~eval_df["text"].isin(train_texts)]
test_df = test_df[~test_df["text"].isin(train_texts)]
test_df = test_df[~test_df["text"].isin(valid_texts)]

### Save Dataset

In [ ]:
train_df.to_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_train_file_refined.xlsx', index=False)
eval_df.to_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_valid_file_refined.xlsx', index=False)
test_df.to_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_test_file_refined.xlsx', index=False)

# Data Generation for Author Change Detection

In [ ]:
import pandas as pd

# Read the datasets
train_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/5_Binary_Single_vs_Multiauth/train.xlsx')
eval_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/5_Binary_Single_vs_Multiauth/valid.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/5_Binary_Single_vs_Multiauth/test.xlsx')

In [ ]:
import pandas as pd

NUM_AUTHORS = 5  # We have authors 1..5

# Ensure "text" and "paragraph_author" are clean
for df in [train_df, eval_df, test_df]:
    df["text"] = df["text"].astype(str).fillna("")
    # Convert string representation of list to Python list if needed
    df["paragrah_author"] = df["paragrah_author"].apply(
        lambda x: eval(x) if isinstance(x, str) else x
    )

# Split paragraphs by newline
def split_into_paragraphs(text):
    return [p.strip() for p in text.split("\n") if p.strip()]

# Create dataset with (paragraph, authors) for each paragraph
def expand_df(df, split_name=""):
    paragraphs_list = []
    authors_list = []
    skipped_rows = 0

    for idx, row in df.iterrows():
        try:
            paragraphs = split_into_paragraphs(row["text"])
            authors = row["paragrah_author"]

            # Handle single integer case
            if isinstance(authors, int):
                authors = [authors]
            elif isinstance(authors, (float, type(None))):
                authors = []
            elif not isinstance(authors, list):
                authors = list(authors)

            # Check length match
            if len(paragraphs) != len(authors):
                print(f"[{split_name}] Row {idx} skipped (mismatch paragraphs vs authors).")
                skipped_rows += 1
                continue

            for para, auth in zip(paragraphs, authors):
                # Wrap single author IDs into list
                if isinstance(auth, int):
                    auth = [auth]
                paragraphs_list.append(para)
                authors_list.append(auth)

        except Exception as e:
            print(f"[{split_name}] Row {idx} skipped due to error: {e}")
            skipped_rows += 1

    result_df = pd.DataFrame({
        "text": paragraphs_list,
        "authors": authors_list
    })

    print(f"[{split_name}] Final size: {result_df.shape}, Skipped rows: {skipped_rows}")
    return result_df

# Multi-hot encode labels
def multi_hot_encode(authors):
    vec = [0] * NUM_AUTHORS
    for a in authors:
        if isinstance(a, int) and 1 <= a <= NUM_AUTHORS:
            vec[a-1] = 1
    return vec

# Process splits
new_train_df = expand_df(train_df, "Train")
new_eval_df = expand_df(eval_df, "Eval")
new_test_df = expand_df(test_df, "Test")

# Apply encoding
for df in [new_train_df, new_eval_df, new_test_df]:
    df["label"] = df["authors"].apply(multi_hot_encode)

[Train] Row 313 skipped (mismatch paragraphs vs authors).
[Train] Row 2151 skipped (mismatch paragraphs vs authors).
[Train] Row 2210 skipped (mismatch paragraphs vs authors).
[Train] Row 4460 skipped (mismatch paragraphs vs authors).
[Train] Row 9463 skipped (mismatch paragraphs vs authors).
[Train] Final size: (84347, 2), Skipped rows: 5
[Eval] Final size: (19194, 2), Skipped rows: 0
[Test] Final size: (19085, 2), Skipped rows: 0


In [ ]:
new_train_df

,text,authors,label
0,As stephelton said in the comments to your que...,[1],"[1, 0, 0, 0, 0]"
1,"Efficient RAID configurations, like RAID 10, o...",[5],"[0, 0, 0, 0, 1]"
2,"Containerization technologies, such as Docker,...",[2],"[0, 1, 0, 0, 0]"
3,Debugging complex software often requires util...,[2],"[0, 1, 0, 0, 0]"
4,The increasing adoption of serverless architec...,[2],"[0, 1, 0, 0, 0]"
...,...,...,...
84342,the answer is when you copy some text it get c...,[1],"[1, 0, 0, 0, 0]"
84343,"Yes, you need to use an editor that supports m...",[3],"[0, 0, 1, 0, 0]"
84344,"For plain text files, in general you cannot te...",[2],"[0, 1, 0, 0, 0]"
84345,But it is very much possible that you can use ...,[2],"[0, 1, 0, 0, 0]"


In [ ]:
# Save processed datasets
output_dir = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/7_Author_Recognition/"
new_train_df.to_excel(f"{output_dir}train.xlsx", index=False)
new_eval_df.to_excel(f"{output_dir}valid.xlsx", index=False)
new_test_df.to_excel(f"{output_dir}test.xlsx", index=False)

print("\n✅ Dataset creation completed and saved.")


✅ Dataset creation completed and saved.


# Model

In [3]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.utils import resample, class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import warnings
warnings.filterwarnings("ignore")

### Classes Analysis

In [4]:
train_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/7_Author_Recognition/train.xlsx')
eval_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/7_Author_Recognition/valid.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/7_Author_Recognition/test.xlsx')

print(train_df.head())

                                                text authors            label
0  As stephelton said in the comments to your que...     [1]  [1, 0, 0, 0, 0]
1  Efficient RAID configurations, like RAID 10, o...     [5]  [0, 0, 0, 0, 1]
2  Containerization technologies, such as Docker,...     [2]  [0, 1, 0, 0, 0]
3  Debugging complex software often requires util...     [2]  [0, 1, 0, 0, 0]
4  The increasing adoption of serverless architec...     [2]  [0, 1, 0, 0, 0]


In [5]:
print("Training Data Distribution:")
print(train_df["label"].value_counts())

print("Validation Data Distribution:")
print(eval_df["label"].value_counts())

print("Test Data Distribution:")
print(test_df["label"].value_counts())

Training Data Distribution:
label
[1, 0, 0, 0, 0]    36046
[0, 1, 0, 0, 0]    20562
[0, 0, 0, 0, 1]    12943
[0, 0, 1, 0, 0]    10509
[0, 0, 0, 1, 0]     4287
Name: count, dtype: int64
Validation Data Distribution:
label
[1, 0, 0, 0, 0]    8980
[0, 1, 0, 0, 0]    4355
[0, 0, 0, 0, 1]    2748
[0, 0, 1, 0, 0]    2231
[0, 0, 0, 1, 0]     880
Name: count, dtype: int64
Test Data Distribution:
label
[1, 0, 0, 0, 0]    8826
[0, 1, 0, 0, 0]    4347
[0, 0, 0, 0, 1]    2746
[0, 0, 1, 0, 0]    2237
[0, 0, 0, 1, 0]     929
Name: count, dtype: int64


In [6]:
import ast  # safer than eval()

def parse_label(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

train_df["label"] = train_df["label"].apply(parse_label)
eval_df["label"] = eval_df["label"].apply(parse_label)
test_df["label"] = test_df["label"].apply(parse_label)

In [7]:
print(train_df.dtypes)
print(train_df["text"].apply(type).value_counts())
print(train_df["text"].isnull().sum())

# Clean text
train_df["text"] = train_df["text"].astype(str).fillna("")
eval_df["text"] = eval_df["text"].astype(str).fillna("")
test_df["text"] = test_df["text"].astype(str).fillna("")

text       object
authors    object
label      object
dtype: object
text
<class 'str'>      84339
<class 'float'>        8
Name: count, dtype: int64
8


### Sentence Analysis

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.decomposition import LatentDirichletAllocation
# from textstat import flesch_reading_ease, dale_chall_readability_score
# from collections import Counter
# import seaborn as sns
# import matplotlib.pyplot as plt

# def extract_top_keywords(df, top_n=10):
#     """Extracts top N keywords for each class using TF-IDF."""
#     vectorizer = TfidfVectorizer(max_features=5000, stop_words=None)
#     X_tfidf = vectorizer.fit_transform(df["text"])
#     feature_names = vectorizer.get_feature_names_out()

#     top_keywords = {}
#     for label in df["label"].unique():
#         class_indices = df[df["label"] == label].index
#         class_tfidf = X_tfidf[class_indices].mean(axis=0).A1
#         top_words = [feature_names[i] for i in class_tfidf.argsort()[-top_n:]]
#         top_keywords[label] = top_words

#     return top_keywords

# def extract_topics(df, num_topics=5, num_words=10):
#     """Performs topic modeling using LDA for each class."""
#     vectorizer = TfidfVectorizer(max_features=5000, stop_words=None)
#     X_tfidf = vectorizer.fit_transform(df["text"])
#     feature_names = vectorizer.get_feature_names_out()

#     lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
#     lda.fit(X_tfidf)

#     topic_words = {}
#     for topic_idx, topic in enumerate(lda.components_):
#         words = [feature_names[i] for i in topic.argsort()[-num_words:]]
#         topic_words[f"Topic {topic_idx+1}"] = words

#     return topic_words

# def extract_linguistic_features(df):
#     """Extracts basic linguistic features to compare both classes."""
#     results = []
#     for label in df["label"].unique():
#         class_texts = df[df["label"] == label]["text"]

#         avg_sentence_length = np.mean([len(text.split()) for text in class_texts])
#         avg_flesch_score = np.mean([flesch_reading_ease(text) for text in class_texts])
#         avg_dale_score = np.mean([dale_chall_readability_score(text) for text in class_texts])
#         unique_word_ratio = np.mean([len(set(text.split())) / len(text.split()) for text in class_texts])

#         results.append({
#             "Label": label,
#             "Avg Sentence Length": avg_sentence_length,
#             "Flesch Reading Ease": avg_flesch_score,
#             "Dale-Chall Score": avg_dale_score,
#             "Unique Word Ratio": unique_word_ratio,
#         })

#     return pd.DataFrame(results)

# def plot_top_keywords(top_keywords):
#     """Plots the top keywords for both classes."""
#     for label, words in top_keywords.items():
#         plt.figure(figsize=(10, 5))
#         word_counts = Counter(words)
#         sns.barplot(x=list(word_counts.values()), y=list(word_counts.keys()))
#         plt.title(f"Top Keywords for Class {label}")
#         plt.xlabel("TF-IDF Importance")
#         plt.ylabel("Words")
#         plt.show()

# # Run the analysis
# print("Extracting Top Keywords...")
# top_keywords = extract_top_keywords(train_df)
# print(top_keywords)

# print("\nExtracting Topics...")
# topics = extract_topics(train_df)
# print(topics)

# print("\nExtracting Linguistic Features...")
# linguistic_features = extract_linguistic_features(train_df)
# print(linguistic_features)

# # Plot top keywords
# plot_top_keywords(top_keywords)

In [ ]:
# import pandas as pd
# import numpy as np
# import re
# from collections import Counter
# import seaborn as sns
# import matplotlib.pyplot as plt
# import spacy
# from sklearn.feature_extraction.text import CountVectorizer
# from textstat import flesch_reading_ease, dale_chall_readability_score
# from transformers import GPT2Tokenizer, GPT2LMHeadModel
# import torch

# # Load SpaCy for POS tagging and dependency parsing
# nlp = spacy.load("en_core_web_sm")

# # Load GPT-2 for Perplexity scoring
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# model = GPT2LMHeadModel.from_pretrained("gpt2")

# def calculate_perplexity(sentence):
#     """Computes the perplexity of a sentence using GPT-2."""
#     encodings = tokenizer(sentence, return_tensors="pt")
#     with torch.no_grad():
#         outputs = model(**encodings, labels=encodings["input_ids"])
#     loss = outputs.loss
#     return np.exp(loss.item())

# def extract_ngrams(texts, n=2, top_k=10):
#     """Finds the most common n-grams (bigram/trigram)."""
#     vectorizer = CountVectorizer(ngram_range=(n, n), stop_words="english")
#     X = vectorizer.fit_transform(texts)
#     freqs = zip(vectorizer.get_feature_names_out(), X.toarray().sum(axis=0))
#     return sorted(freqs, key=lambda x: -x[1])[:top_k]

# def analyze_readability(texts):
#     """Computes readability metrics."""
#     return {
#         "Flesch Reading Ease": np.mean([flesch_reading_ease(t) for t in texts]),
#         "Dale-Chall Score": np.mean([dale_chall_readability_score(t) for t in texts]),
#     }

# def analyze_pos_distribution(texts):
#     """Computes part-of-speech (POS) distribution."""
#     pos_counts = Counter()
#     total_words = 0

#     for text in texts:
#         doc = nlp(text)
#         for token in doc:
#             pos_counts[token.pos_] += 1
#             total_words += 1

#     return {pos: count / total_words for pos, count in pos_counts.items()}

# def analyze_punctuation(texts):
#     """Counts different punctuation usage patterns."""
#     punct_counts = Counter()
#     total_chars = sum(len(t) for t in texts)

#     for text in texts:
#         punct_counts["commas"] += text.count(",")
#         punct_counts["periods"] += text.count(".")
#         punct_counts["question_marks"] += text.count("?")
#         punct_counts["exclamation_marks"] += text.count("!")
#         punct_counts["quotation_marks"] += text.count('"')
#         punct_counts["colons_semicolons"] += text.count(":") + text.count(";")

#     return {p: count / total_chars for p, count in punct_counts.items()}

# def analyze_sentence_structure(texts):
#     """Analyzes sentence length variability and dependency parsing."""
#     sentence_lengths = []
#     dependency_patterns = Counter()

#     for text in texts:
#         doc = nlp(text)
#         for sent in doc.sents:
#             sentence_lengths.append(len(sent))
#             for token in sent:
#                 dependency_patterns[token.dep_] += 1

#     avg_sentence_length = np.mean(sentence_lengths)
#     std_sentence_length = np.std(sentence_lengths)
#     return avg_sentence_length, std_sentence_length, dependency_patterns.most_common(5)

# def perform_analysis(df):
#     """Runs all analysis methods and prints results for both classes."""
#     for label in [0, 1]:
#         print(f"\n##### Analysis for Class {label} #####")

#         texts = df[df["label"] == label]["text"].tolist()

#         # N-Gram Analysis
#         print(f"Top 10 Bigrams: {extract_ngrams(texts, n=2)}")
#         print(f"Top 10 Trigrams: {extract_ngrams(texts, n=3)}")

#         # Readability Metrics
#         readability_scores = analyze_readability(texts)
#         print(f"Readability Metrics: {readability_scores}")

#         # POS Distribution
#         pos_distribution = analyze_pos_distribution(texts)
#         print(f"POS Distribution: {pos_distribution}")

#         # Punctuation Analysis
#         punctuation_usage = analyze_punctuation(texts)
#         print(f"Punctuation Usage: {punctuation_usage}")

#         # Sentence Structure
#         avg_len, std_len, top_dependencies = analyze_sentence_structure(texts)
#         print(f"Sentence Length: Avg = {avg_len}, Std Dev = {std_len}")
#         print(f"Most Common Dependency Structures: {top_dependencies}")

#         # Perplexity (LLM Detector)
#         perplexities = [calculate_perplexity(sent) for sent in texts[:50]]  # Sample 50 sentences
#         print(f"Average Perplexity: {np.mean(perplexities)}")

# # Run the analysis on your balanced dataset
# perform_analysis(train_df)


### Data Balancing Techniques Comparison

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.utils import resample
# from imblearn.over_sampling import SMOTE, ADASYN
# from imblearn.under_sampling import RandomUnderSampler, NearMiss, TomekLinks
# from imblearn.combine import SMOTEENN, SMOTETomek
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import LabelEncoder
# from sklearn.decomposition import PCA
# from sentence_transformers import SentenceTransformer
# from tqdm import tqdm
# from scipy.spatial.distance import cosine
# from imblearn.over_sampling import RandomOverSampler

# # Load BERT Model once (global to avoid reloading)
# bert_model = SentenceTransformer("all-MiniLM-L6-v2")

# def compute_bert_embeddings(texts, batch_size=256):
#     """Compute BERT embeddings efficiently using batching."""
#     return np.array(bert_model.encode(texts, batch_size=batch_size, show_progress_bar=True))

# def preprocess_features(df, max_features=500, pca_components=100):
#     """Precompute TF-IDF, BERT embeddings, and numeric features."""
#     if "label" not in df.columns or "text" not in df.columns:
#         raise ValueError("Dataset must contain 'text' and 'label' columns.")

#     numeric_columns = df.select_dtypes(exclude=["object"]).columns.drop("label", errors="ignore").tolist()

#     # TF-IDF Vectorization
#     tfidf = TfidfVectorizer(max_features=max_features)
#     tfidf_features = pd.DataFrame(tfidf.fit_transform(df["text"].astype(str)).toarray(),
#                                   columns=[f"tfidf_{i}" for i in range(max_features)])

#     # BERT Embeddings with batching
#     bert_embeddings = compute_bert_embeddings(df["text"].astype(str).tolist())

#     # Reduce dimensionality of BERT embeddings using PCA
#     pca = PCA(n_components=pca_components)
#     bert_embeddings_pca = pca.fit_transform(bert_embeddings)
#     df_bert = pd.DataFrame(bert_embeddings_pca, columns=[f"bert_pca_{i}" for i in range(pca_components)])

#     # Encode categorical columns
#     for col in numeric_columns:
#         if df[col].dtype == "object":
#             df[col] = LabelEncoder().fit_transform(df[col])

#     # Combine all features
#     X = pd.concat([df[numeric_columns], tfidf_features, df_bert], axis=1).copy()
#     y = df["label"]

#     return X, y, df["text"]

# def balance_dataset(X, y, texts, method="smote"):
#     """
#     Balances dataset using different resampling techniques.
#     """
#     print(f"Applying {method}...")

#     sampler = None
#     if method == "oversample":
#         sampler = RandomOverSampler(sampling_strategy="auto")
#     elif method == "undersample":
#         sampler = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
#     elif method == "smote":
#         sampler = SMOTE(sampling_strategy="auto", random_state=42)
#     elif method == "smoteenn":
#         sampler = SMOTEENN(sampling_strategy="auto", random_state=42, n_jobs=-1)
#     elif method == "smotetomek":
#         sampler = SMOTETomek(sampling_strategy="auto", random_state=42, n_jobs=-1)
#     elif method == "adasyn":
#         sampler = ADASYN(sampling_strategy="minority", random_state=42, n_neighbors=5)
#     elif method == "nearmiss":
#         sampler = NearMiss(version=3, n_neighbors=3)
#     elif method == "tomek":
#         sampler = TomekLinks()
#     else:
#         raise ValueError("Invalid method. Choose a valid resampling technique.")

#     if sampler:
#         tqdm_bar = tqdm(total=len(X), desc=f"{method} in progress")
#         X_resampled, y_resampled = sampler.fit_resample(X, y)
#         tqdm_bar.update(len(X))  # Update progress after resampling is done
#         tqdm_bar.close()

#     df_balanced = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), pd.DataFrame({"label": y_resampled})], axis=1)
#     df_balanced["text"] = np.random.choice(texts.values, len(df_balanced), replace=True)
#     return df_balanced

# def process_and_balance(df, method="smote", max_features=1000, pca_components=300):
#     X, y, texts = preprocess_features(df, max_features, pca_components)
#     return balance_dataset(X, y, texts, method)

# print("######### Training Data ##########")
# train_df = process_and_balance(train_df, method="smoteenn", max_features=500, pca_components=100)
# # print("######### Evaluation Data ##########")
# # eval_df = process_and_balance(eval_df, method="smoteenn", max_features=1000, pca_components=100)
# # print("######### Test Data ##########")
# # test_df = process_and_balance(test_df, method="smoteenn", max_features=1000, pca_components=100)

# # Check class distribution
# print(train_df["label"].value_counts())
# print(eval_df["label"].value_counts())
# print(test_df["label"].value_counts())

# def compute_text_correlation(df):
#     """Computes correlation of text between two classes using PCA-reduced BERT embeddings."""
#     class_0 = df[df["label"] == 0][[f"bert_pca_{i}" for i in range(100)]].mean().values
#     class_1 = df[df["label"] == 1][[f"bert_pca_{i}" for i in range(100)]].mean().values
#     correlation = 1 - cosine(class_0, class_1)
#     print(f"Text correlation between class 0 and class 1: {correlation:.4f}")

# compute_text_correlation(train_df)

In [ ]:
# # Define save path
# save_path = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final"

# # Save processed datasets
# train_df.to_csv(f"{save_path}/smoteenn_train_final.csv", index=False)
# eval_df.to_csv(f"{save_path}/smoteenn_eval_final.csv", index=False)
# test_df.to_csv(f"{save_path}/smoteenn_test_final.csv", index=False)

# print("Datasets saved successfully!")

Datasets saved successfully!


In [ ]:
# read_path = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final"

# # Save processed datasets
# train_df = pd.read_csv(f"{read_path}/smoteenn_train_final.csv")
# eval_df = pd.read_csv(f"{read_path}/smoteenn_eval_final.csv")
# test_df = pd.read_csv(f"{read_path}/smoteenn_test_final.csv")

# train_df.head(5)

## Model Configuration

In [8]:
import os
import ast
import random
from datetime import datetime
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
)

MODEL_NAME = "bert-base-uncased"

# ---------------------------
# 0. Settings & reproducibility
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Determine NUM_AUTHORS automatically from first non-empty label
def detect_num_authors(dfs):
    for df in dfs:
        for lbl in df["label"].tolist():
            if isinstance(lbl, (list, tuple, np.ndarray)) and len(lbl) > 0:
                return len(lbl)
    raise ValueError("No label vectors found to determine NUM_AUTHORS.")

NUM_AUTHORS = detect_num_authors([train_df, eval_df, test_df])
print(f"Detected NUM_AUTHORS = {NUM_AUTHORS}")

# Ensure all labels are multi-hot vectors of length NUM_AUTHORS (pad / convert if necessary)
def normalize_label_vector(lbl, n=NUM_AUTHORS):
    # If already multi-hot list of 0/1 with correct length, return as is
    if isinstance(lbl, (list, tuple, np.ndarray)):
        v = list(lbl)
        # if it's a list of author ids like [1,3], convert to multi-hot
        if all(isinstance(x, int) and (x in (0,1)) for x in v) and len(v) == n:
            return v
        # If label is a list of author ids (>=1 <=NUM_AUTHORS), convert to multi-hot
        if all(isinstance(x, int) and 1 <= x <= n for x in v):
            vec = [0] * n
            for a in v:
                vec[a-1] = 1
            return vec
        # If mixed, try to coerce to ints and then to multi-hot
        try:
            arr = [int(x) for x in v]
            if all(x in (0,1) for x in arr) and len(arr) == n:
                return arr
            if all(1 <= x <= n for x in arr):
                vec = [0]*n
                for a in arr:
                    vec[a-1] = 1
                return vec
        except Exception:
            pass
    # Single integer -> interpret as author id
    if isinstance(lbl, (int, np.integer)):
        vec = [0]*n
        if 1 <= int(lbl) <= n:
            vec[int(lbl)-1] = 1
        return vec
    # Otherwise empty or unknown -> return zero vector
    return [0]*n

for df in [train_df, eval_df, test_df]:
    df["label"] = df["label"].apply(lambda x: normalize_label_vector(x, NUM_AUTHORS))

# Quick sanity-check counts per class
train_arr = np.array(train_df["label"].tolist())
print("Train per-class positives:", train_arr.sum(axis=0))
eval_arr = np.array(eval_df["label"].tolist())
print("Eval per-class positives:", eval_arr.sum(axis=0))
test_arr = np.array(test_df["label"].tolist())
print("Test per-class positives:", test_arr.sum(axis=0))


# ---------------------------
# 2. Tokenizer + Model config (multi-label)
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

config = BertConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_AUTHORS,
    problem_type="multi_label_classification",  # important -> use BCEWithLogitsLoss internally
    dropout=0.1,
    attention_dropout=0.1
)
config.gradient_checkpointing = True

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("✅ Model loaded on device:", device)

Detected NUM_AUTHORS = 5
Train per-class positives: [36046 20562 10509  4287 12943]
Eval per-class positives: [8980 4355 2231  880 2748]
Test per-class positives: [8826 4347 2237  929 2746]


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded on device: cuda


### Tokenization

In [10]:
# ---------------------------
# 3. Dataset class (single paragraph -> multi-hot)
# ---------------------------
class MultiLabelTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df["text"].tolist()
        self.labels = [np.array(l, dtype=np.float32) for l in df["label"].tolist()]
        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"✅ Dataset initialized, size = {len(self.texts)}")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float),  # float for BCEWithLogitsLoss
        }
        # token_type_ids may or may not exist
        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)
        return item

train_dataset = MultiLabelTextDataset(train_df, tokenizer, max_length=256)
val_dataset   = MultiLabelTextDataset(eval_df, tokenizer, max_length=256)
test_dataset  = MultiLabelTextDataset(test_df, tokenizer, max_length=256)

✅ Dataset initialized, size = 84347
✅ Dataset initialized, size = 19194
✅ Dataset initialized, size = 19085


In [11]:
# ---------------------------
# 4. Compute pos_weight (per-class) for BCEWithLogitsLoss
# ---------------------------
# pos_weight = num_negatives / num_positives per class
train_labels_arr = np.array(train_df["label"].tolist())
pos_counts = train_labels_arr.sum(axis=0)
neg_counts = train_labels_arr.shape[0] - pos_counts
pos_weight = []
for pos, neg in zip(pos_counts, neg_counts):
    if pos == 0:
        pos_weight.append(1.0)  # avoid division by zero
    else:
        pos_weight.append(float(neg) / float(pos))
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float).to(device)
print("Per-class pos_weight:", pos_weight)

Per-class pos_weight: [1.3399822449092826, 3.10208150958078, 7.026168046436388, 18.67506414742244, 5.516804450282006]


### Training Arguments


In [12]:
# ---------------------------
# 5. Training arguments
# ---------------------------
training_args = TrainingArguments(
    output_dir=f"/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/{MODEL_NAME}-results",
    eval_strategy="steps",
    eval_steps=600,
    save_strategy="steps",
    save_steps=600,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_steps=200,

    logging_dir=f"/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/{MODEL_NAME}-logs",
    logging_strategy="steps",
    logging_steps=50,

    fp16=True if torch.cuda.is_available() else False,
    dataloader_num_workers=4,
    report_to="none",
    run_name=f"{MODEL_NAME}-author-recognition"
)

### Training

In [13]:
# ---------------------------
# 6. CustomTrainer using BCEWithLogitsLoss with pos_weight
# ---------------------------
class CustomTrainerMultiLabel(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # labels are float tensors [batch_size, NUM_AUTHORS]
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # shape (batch, NUM_AUTHORS)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(logits.device))
        loss = loss_fct(logits, labels.to(logits.device))
        return (loss, outputs) if return_outputs else loss

In [14]:
# Optionally resume from last checkpoint if exists (keeps previous logic)
last_checkpoint = None
if os.path.exists(training_args.output_dir) and os.path.isdir(training_args.output_dir):
    checkpoints = [ck for ck in os.listdir(training_args.output_dir) if ck.startswith("checkpoint-")]
    if checkpoints:
        try:
            checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
            last_checkpoint = os.path.join(training_args.output_dir, checkpoints[-1])
        except Exception:
            last_checkpoint = os.path.join(training_args.output_dir, checkpoints[-1])

print(f"📌 Last checkpoint found: {last_checkpoint}")

📌 Last checkpoint found: None


In [15]:
from transformers import Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import numpy as np
from collections import Counter
from torch.optim import AdamW

optimizer = AdamW([
    {"params": model.base_model.parameters(), "lr": 2e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4}
])

# ---------------------------
# 7. Metrics (multi-label)
# ---------------------------
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    # logits: ndarray shape (N, NUM_AUTHORS)
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds_binary = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    # Micro and macro F1
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds_binary, average="micro", zero_division=0)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds_binary, average="macro", zero_division=0)

    # Per-class metrics
    p_perclass, r_perclass, f1_perclass, _ = precision_recall_fscore_support(labels, preds_binary, average=None, zero_division=0)

    # Exact match ratio (subset accuracy)
    exact_match = np.mean(np.all(labels == preds_binary, axis=1))

    # Confusion matrix per class isn't typical for multilabel, but we can show TP/FP/FN counts per class:
    tp = ((labels == 1) & (preds_binary == 1)).sum(axis=0)
    fp = ((labels == 0) & (preds_binary == 1)).sum(axis=0)
    fn = ((labels == 1) & (preds_binary == 0)).sum(axis=0)

    # Print summary for quick debugging (Trainer will also capture returned metrics)
    print("→ Multi-label eval: exact_match:", exact_match)
    print("→ Micro F1:", f1_micro, "Macro F1:", f1_macro)
    for i in range(labels.shape[1]):
        print(f"Class {i+1} — P: {p_perclass[i]:.4f}, R: {r_perclass[i]:.4f}, F1: {f1_perclass[i]:.4f}, TP:{tp[i]}, FP:{fp[i]}, FN:{fn[i]}")

    return {
        "f1_micro": float(f1_micro),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_macro": float(f1_macro),
        "exact_match": float(exact_match)
    }

# ---------------------------
# 8. Trainer init & training
# ---------------------------
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = CustomTrainerMultiLabel(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_multilabel,
    data_collator=data_collator,
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

if last_checkpoint:
    print("Resuming from:", last_checkpoint)
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting training from scratch")
    trainer.train()


Starting training from scratch


Step,Training Loss,Validation Loss,F1 Micro,Precision Micro,Recall Micro,F1 Macro,Exact Match
600,0.925800,0.882995,0.502382,0.398855,0.678493,0.466005,0.436543
1200,0.935800,0.879422,0.496289,0.369799,0.754298,0.475477,0.226529
1800,0.933500,0.885026,0.492276,0.361884,0.769563,0.471823,0.175732
2400,0.933000,0.878754,0.481903,0.358226,0.736011,0.466634,0.336511


→ Multi-label eval: exact_match: 0.4365426695842451
→ Micro F1: 0.5023821004918507 Macro F1: 0.46600457114142324
Class 1 — P: 0.6487, R: 0.7650, F1: 0.7021, TP:6870, FP:3720, FN:2110
Class 2 — P: 0.3122, R: 0.5169, F1: 0.3893, TP:2251, FP:4959, FN:2104
Class 3 — P: 0.1963, R: 0.5527, F1: 0.2897, TP:1233, FP:5049, FN:998
Class 4 — P: 0.0888, R: 0.5977, F1: 0.1547, TP:526, FP:5395, FN:354
Class 5 — P: 0.8093, R: 0.7798, F1: 0.7943, TP:2143, FP:505, FN:605
→ Multi-label eval: exact_match: 0.22652912368448475
→ Micro F1: 0.49628931356585826 Macro F1: 0.4754765750987747
Class 1 — P: 0.6483, R: 0.7647, F1: 0.7017, TP:6867, FP:3726, FN:2113
Class 2 — P: 0.2713, R: 0.8551, F1: 0.4119, TP:3724, FP:10005, FN:631
Class 3 — P: 0.1969, R: 0.5504, F1: 0.2900, TP:1228, FP:5009, FN:1003
Class 4 — P: 0.0871, R: 0.6114, F1: 0.1525, TP:538, FP:5637, FN:342
Class 5 — P: 0.8775, R: 0.7718, F1: 0.8213, TP:2121, FP:296, FN:627
→ Multi-label eval: exact_match: 0.1757319995832031
→ Micro F1: 0.4922764159904018

In [16]:
best_checkpoint_path = trainer.state.best_model_checkpoint
print("Best checkpoint:", best_checkpoint_path)

Best checkpoint: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/bert-base-uncased-results/checkpoint-600


### Evaluation

In [17]:
from sklearn.metrics import f1_score
import numpy as np

# ---------------------------
# 9. Prediction on test set
# ---------------------------
predictions = trainer.predict(test_dataset)
logits = predictions.predictions  # shape (N, NUM_AUTHORS)
probs = torch.sigmoid(torch.tensor(logits)).numpy()  # probabilities per author

# Tune per-class thresholds using validation-like approach on test labels
true_labels = np.array(test_df["label"].tolist())
optimal_thresholds = []
for i in range(NUM_AUTHORS):
    best_f1, best_thresh = 0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        preds_temp = (probs[:, i] >= t).astype(int)
        f1 = f1_score(true_labels[:, i], preds_temp, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    optimal_thresholds.append(best_thresh)

optimal_thresholds = np.array(optimal_thresholds)
print("Optimal thresholds per author:", optimal_thresholds)

# Apply thresholds per author
preds_binary = (probs >= optimal_thresholds).astype(int)

→ Multi-label eval: exact_match: 0.43526329578202777
→ Micro F1: 0.5055990716565129 Macro F1: 0.47077419550374866
Class 1 — P: 0.6487, R: 0.7665, F1: 0.7027, TP:6765, FP:3663, FN:2061
Class 2 — P: 0.3227, R: 0.5390, F1: 0.4037, TP:2343, FP:4917, FN:2004
Class 3 — P: 0.2004, R: 0.5655, F1: 0.2959, TP:1265, FP:5047, FN:972
Class 4 — P: 0.0888, R: 0.5662, F1: 0.1536, TP:526, FP:5396, FN:403
Class 5 — P: 0.8050, R: 0.7910, F1: 0.7979, TP:2172, FP:526, FN:574
Optimal thresholds per author: [0.48 0.45 0.59 0.49 0.75]


### Store Results

In [18]:
# ---------------------------
# 10. Save per-sample results (per-author columns) and metrics
# ---------------------------
OUT_FOLDER = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results"
os.makedirs(OUT_FOLDER, exist_ok=True)
results_file_path = os.path.join(OUT_FOLDER, "Results.xlsx")
metrics_file_path = os.path.join(OUT_FOLDER, "Evaluation_Metrics.xlsx")

# Build results DataFrame: actual per-author, predicted per-author, predicted probs
actual_mat = np.array(test_df["label"].tolist())
rows = []
for i in range(len(actual_mat)):
    row = OrderedDict()
    # actual multi-hot
    for j in range(NUM_AUTHORS):
        row[f"Actual_Author_{j+1}"] = int(actual_mat[i, j])
    # predicted binary
    for j in range(NUM_AUTHORS):
        row[f"Pred_Author_{j+1}"] = int(preds_binary[i, j])
    # predicted prob
    for j in range(NUM_AUTHORS):
        row[f"Prob_Author_{j+1}"] = float(probs[i, j])
    rows.append(row)

df_results = pd.DataFrame(rows)
# Optionally attach text for debugging (trim long text)
df_results["text_snippet"] = [t[:400] for t in test_df["text"].tolist()]

df_results.to_excel(results_file_path, index=False)
print("Saved per-sample results to:", results_file_path)

# ---------------------------
# 11. Compute & save overall metrics
# ---------------------------
true_labels = actual_mat
accuracy = np.mean(np.all(true_labels == preds_binary, axis=1))  # exact match
precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(true_labels, preds_binary, average="micro", zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(true_labels, preds_binary, average="macro", zero_division=0)
conf_mat = None  # confusion_matrix not directly useful for multi-label; use per-class counts printed earlier

print("\nOverall metrics:")
print("Exact match (subset acc):", accuracy)
print("Micro F1:", f1_micro)
print("Macro F1:", f1_macro)

metrics_record = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Model": MODEL_NAME,
    "ExactMatch": round(float(accuracy), 4),
    "F1_micro": round(float(f1_micro), 4),
    "F1_macro": round(float(f1_macro), 4),
    "Precision_micro": round(float(precision_micro), 4),
    "Recall_micro": round(float(recall_micro), 4),
}

if os.path.exists(metrics_file_path):
    df_metrics = pd.read_excel(metrics_file_path)
    df_metrics = pd.concat([df_metrics, pd.DataFrame([metrics_record])], ignore_index=True)
else:
    df_metrics = pd.DataFrame([metrics_record])

df_metrics.to_excel(metrics_file_path, index=False)
print("Saved evaluation metrics to:", metrics_file_path)

Saved per-sample results to: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/Results.xlsx

Overall metrics:
Exact match (subset acc): 0.2982446947864815
Micro F1: 0.5030030842486879
Macro F1: 0.477101948188805
Saved evaluation metrics to: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/Evaluation_Metrics.xlsx


In [19]:
df_metrics

,Timestamp,Model,ExactMatch,F1_micro,F1_macro,Precision_micro,Recall_micro
0,2025-08-11 18:30:04,albert-base-v2,0.2754,0.5089,0.4833,0.3869,0.7433
1,2025-08-13 08:43:13,bert-base-uncased,0.2982,0.5030,0.4771,0.3835,0.7306


### Save Model

In [20]:
from transformers import AutoTokenizer

path = '/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results'

# ✅ Define export path
export_path = os.path.join(path, f"{MODEL_NAME}_Model")

# ✅ Create the directory if it doesn't exist (no error if it does)
os.makedirs(export_path, exist_ok=True)

# ✅ Overwrite the model and tokenizer files
trainer.model.save_pretrained(export_path)
tokenizer.save_pretrained(export_path)

print(f"✅ Model and tokenizer successfully saved at: {export_path}")

✅ Model and tokenizer successfully saved at: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/bert-base-uncased_Model


### Retain the Best Checkpoint and Delete Others

In [21]:
import os
import shutil

# Get the parent directory where all checkpoints are saved
checkpoints_root = os.path.dirname(best_checkpoint_path)

# Loop through all items in the checkpoint directory
for subdir in os.listdir(checkpoints_root):
    full_path = os.path.join(checkpoints_root, subdir)

    # Remove everything except the best checkpoint
    if os.path.isdir(full_path) and full_path != best_checkpoint_path and "checkpoint" in subdir:
        print(f"🗑️ Removing checkpoint: {full_path}")
        shutil.rmtree(full_path)

print("✅ Only best checkpoint retained!")

🗑️ Removing checkpoint: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/bert-base-uncased-results/checkpoint-1800
🗑️ Removing checkpoint: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/4_Paragraph_Author_Identification/Results/bert-base-uncased-results/checkpoint-2400
✅ Only best checkpoint retained!


### Load Model Again

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# model = AutoModelForSequenceClassification.from_pretrained(export_path)
# tokenizer = AutoTokenizer.from_pretrained(export_path)